## BERT 微調流程

> 此 Notebook 保留舊版 Colab 實驗的模型架構與訓練步驟。公開版本已移除執行輸出與環境暫存資訊；完整指標與 split overlap 限制請見 `docs/EXPERIMENTS.md`。執行前請自行在 Colab 工作目錄放置未公開的 `data.csv`。


In [ ]:
import pandas as pd

try:
    df = pd.read_csv(
        "data.csv",
        sep=",",
        names=["text", "label"],
        header=0,
    )
    total_count = len(df)
    label_counts = df["label"].value_counts()

    print("資料統計：")
    print(f"總筆數：{total_count}")
    print("-" * 20)

    count_0 = label_counts.get(0, 0)
    count_1 = label_counts.get(1, 0)
    print(f"標籤 0：{count_0} 筆，占 {(count_0 / total_count * 100):.2f}%")
    print(f"標籤 1：{count_1} 筆，占 {(count_1 / total_count * 100):.2f}%")
except FileNotFoundError:
    print("找不到 data.csv")
except Exception as error:
    print(f"讀取資料時發生錯誤：{error}")


In [ ]:
import pandas as pd
df = pd.read_csv("data.csv")
print(df["label"].isna().sum())
df[df["label"].isna()]


In [ ]:
from sklearn.model_selection import GroupShuffleSplit
import pandas as pd

# 依 text 去除完全重複資料，並以 text 作為 group，避免相同文本跨 split。
df = pd.read_csv("data.csv")
df_unique = df.drop_duplicates(subset=["text"]).reset_index(drop=True)

gss = GroupShuffleSplit(test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(df_unique, groups=df_unique["text"]))

train_df = df_unique.iloc[train_idx].copy()
test_df = df_unique.iloc[test_idx].copy()
train_df["label"] = train_df["label"].astype(int)
test_df["label"] = test_df["label"].astype(int)


## 安裝套件


In [ ]:
!pip install transformers datasets torch scikit-learn

In [ ]:
!pip uninstall -y transformers
!pip install -U transformers accelerate


## Tokenization 與資料前處理


In [ ]:
model_name = "hfl/chinese-roberta-wwm-ext"


In [ ]:
from datasets import Dataset
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, max_length=400)

train_ds = Dataset.from_pandas(train_df).map(tokenize, batched=True)
test_ds = Dataset.from_pandas(test_df).map(tokenize, batched=True)
train_ds.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
test_ds.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])


## 訓練設定


In [ ]:
from transformers import AutoModelForSequenceClassification, TrainingArguments

model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)
args = TrainingArguments(
    output_dir="./bert_output",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    logging_steps=100,
    report_to="none",
)


In [ ]:
from transformers import Trainer, DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    data_collator=data_collator,
)


In [ ]:
trainer.train()


## 評估


In [ ]:
from sklearn.metrics import classification_report

preds = trainer.predict(test_ds)
y_pred = preds.predictions.argmax(axis=1)
y_true = test_df["label"].values

print(classification_report(y_true, y_pred))


## 儲存模型


In [ ]:
model.save_pretrained("./my_model")
tokenizer.save_pretrained("./my_model")

In [ ]:
import shutil
from google.colab import files

shutil.make_archive('my_model', 'zip', 'my_model')
files.download('my_model.zip')